In [0]:
# Configure paths
silver_path = "/Volumes/ingestion/raw_ads/raw/delta/ads_silver_file"
gold_base_path = "/Volumes/ingestion/raw_ads/raw/delta"
checkpoint_base_path = "/Volumes/ingestion/raw_ads/raw/checkpoints"

# read silver
silver_df = (
    spark.readStream
        .format("delta")
        .load(silver_path)
)

# aggregate metrics: impressions, clicks, conversions, spend, revenue
from pyspark.sql.functions import *

campaign_metrics = (
    silver_df
    .withColumn("date", to_date("event_time"))
    .groupBy("campaign_id", "date")
    .agg(
        sum(when(col("event_type") == "ad_impression", 1).otherwise(0)).alias("impressions"),
        sum(when(col("event_type") == "ad_click", 1).otherwise(0)).alias("clicks"),
        sum(when(col("event_type") == "conversion", 1).otherwise(0)).alias("conversions"),
        sum("price_paid").alias("spend"),
        sum("conversion_value").alias("revenue")
    )
)

# write gold table 1: campaign performance
campaign_query = (
    campaign_metrics.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", f"{checkpoint_base_path}/gold_campaign_metrics")
        .start(f"{gold_base_path}/gold_campaign_metrics")
)

# compute conversion funnel
funnel_metrics = (
    silver_df
    .groupBy("campaign_id")
    .agg(
        sum(when(col("event_type") == "ad_impression", 1).otherwise(0)).alias("impressions"),
        sum(when(col("event_type") == "ad_click", 1).otherwise(0)).alias("clicks"),
        sum(when(col("event_type") == "conversion", 1).otherwise(0)).alias("conversions")
    )
    .withColumn("ctr", col("clicks") / col("impressions"))
    .withColumn("cvr", col("conversions") / col("clicks"))
)

# write gold table 2: Ad Funnel Metrics
funnel_query = (
    funnel_metrics.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", f"{checkpoint_base_path}/gold_funnel")
        .start(f"{gold_base_path}/gold_funnel")
)

# compute ROAS
roas_metrics = campaign_metrics.withColumn(
    "roas",
    col("revenue") / col("spend")
)

# write gold table 3: ROAS
roas_query = (
    roas_metrics.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", f"{checkpoint_base_path}/gold_roas")
        .start(f"{gold_base_path}/gold_roas")
)

# measure ads per playback session
session_ads = (
    silver_df
    .filter(col("event_type") == "ad_impression")
    .groupBy("session_id")
    .agg(count("*").alias("ads_served"))
)

# write gold table 4: session ad load
session_query = (
    session_ads.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", f"{checkpoint_base_path}/gold_session_ads")
        .start(f"{gold_base_path}/gold_session_ads")
)

# measure engagement
user_metrics = (
    silver_df
    .groupBy("user_id")
    .agg(
        countDistinct("session_id").alias("sessions"),
        sum(when(col("event_type") == "ad_impression", 1).otherwise(0)).alias("ads_seen"),
        sum(when(col("event_type") == "conversion", 1).otherwise(0)).alias("conversions")
    )
)

# write gold table 5: user metrics
user_query = (
    user_metrics.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", f"{checkpoint_base_path}/gold_user_metrics")
        .start(f"{gold_base_path}/gold_user_metrics")
)